In [78]:
import sys
sys.path.append('/home/jovyan/work/Similarity-Aware-Label-Smoothing')


In [79]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import ResNet18, CifarDenseNet121, TinyEfficientNet
from metrics import calibration_errors, nll_loss, accuracy
import random

## Hyperparams

In [80]:
seed = 4212
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [81]:
dataset = "cifar100"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler(device="cuda")
lr = 0.1
epochs = 200
num_classes = 100
epsilon = 0.02
K = 5

## Training Utils

### Label Smoothing

In [82]:
path = f"Similarity-Aware-Label-Smoothing/scores/4_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=K, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [83]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):
    classwise_target = classwise_target.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            targets = classwise_target[y]

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(x)
                loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        if epoch % 10 == 9 or epoch >= 195:
            test_acc = accuracy(model, test_loader)
            print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [84]:
classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target[0])

tensor([0.9800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0038,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0038, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0034,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0052, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000], device='cuda:0')


In [85]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [86]:
model = CifarDenseNet121(num_classes=num_classes).to(device)
model = torch.compile(model, mode="max-autotune")
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

Epoch 10/200 | Loss: 2.0349 | Test Acc: 0.4004 | Top-5 Test Acc: 0.7191
Epoch 20/200 | Loss: 1.7619 | Test Acc: 0.4775 | Top-5 Test Acc: 0.7756
Epoch 30/200 | Loss: 1.6382 | Test Acc: 0.4434 | Top-5 Test Acc: 0.7397
Epoch 40/200 | Loss: 1.5589 | Test Acc: 0.5071 | Top-5 Test Acc: 0.8152
Epoch 50/200 | Loss: 1.4894 | Test Acc: 0.4559 | Top-5 Test Acc: 0.7614
Epoch 60/200 | Loss: 1.4208 | Test Acc: 0.5331 | Top-5 Test Acc: 0.8228
Epoch 70/200 | Loss: 1.3471 | Test Acc: 0.5443 | Top-5 Test Acc: 0.8292
Epoch 80/200 | Loss: 1.2729 | Test Acc: 0.5523 | Top-5 Test Acc: 0.8362
Epoch 90/200 | Loss: 1.1581 | Test Acc: 0.5556 | Top-5 Test Acc: 0.8332
Epoch 100/200 | Loss: 1.0658 | Test Acc: 0.5882 | Top-5 Test Acc: 0.8505
Epoch 110/200 | Loss: 0.9279 | Test Acc: 0.5794 | Top-5 Test Acc: 0.8476
Epoch 120/200 | Loss: 0.8019 | Test Acc: 0.5981 | Top-5 Test Acc: 0.8528
Epoch 130/200 | Loss: 0.6381 | Test Acc: 0.5954 | Top-5 Test Acc: 0.8430
Epoch 140/200 | Loss: 0.4814 | Test Acc: 0.6031 | Top-5 Test

In [87]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)

        logits_list.append(logits.float())  
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)

print("ECE:", calibration_errors(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))

test_acc = accuracy(model, test_loader)  
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


ECE: (0.05361102521419525, 0.13728994131088257)
NLL: 1.3047410249710083
Top-1 Test Acc: 0.6694 | Top-5 Test Acc: 0.8742


In [88]:
PATH = f"vae4_{'0'+str(epsilon).removeprefix("0.")}_{K}_{seed}.pth"
# PATH = f"ls_{'0'+str(epsilon).removeprefix("0.")}_{seed}.pth"
torch.save(model._orig_mod.state_dict(), PATH)